## Deep Research with MCP

In [ ]:
import os
from contextlib import AsyncExitStack
from datetime import datetime
from dotenv import load_dotenv
from IPython.display import Markdown, display
from agents import Agent, OpenAIChatCompletionsModel, Runner, trace
from agents.mcp import MCPServerStdio
from openai import AsyncOpenAI

load_dotenv(override=True)

In [ ]:
SESSION_TIMEOUT = 60
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_MODEL = "openai/gpt-4o-mini"
SERPER_API_KEY = os.getenv("SERPER_API_KEY")

if not OPENROUTER_API_KEY:
    raise EnvironmentError("Please set OPENROUTER_API_KEY in .env")
if not SERPER_API_KEY:
    raise EnvironmentError("Please set SERPER_API_KEY in .env")

openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)
llm_model = OpenAIChatCompletionsModel(
    model=OPENROUTER_MODEL, openai_client=openrouter_client
)


In [ ]:
fetch_params = {
    "command": "uvx",
    "args": ["serper-mcp-server"],
    "env": {"SERPER_API_KEY": SERPER_API_KEY},
}
async with MCPServerStdio(
    params=fetch_params, client_session_timeout_seconds=SESSION_TIMEOUT
) as server:
    tools = await server.list_tools()
tools

In [ ]:
def research_instructions() -> str:
    """
    Generate research instructions for the agent.
    """
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    web_search_prompt = (
        "You have web search tools. Run one or more searches to find candidate pages. "
        "Select 3–5 distinct, authoritative URLs (prefer primary sources, official docs, or reputable outlets). "
        "Do not rely only on search snippets; you must fetch pages next.\n\n"
    )
    return f"""You are a careful research assistant. Current datetime: {now}

Workflow:
{web_search_prompt}
You have a fetch tool. For each URL you will use in the answer, fetch the page content.
If a fetch response is truncated, call fetch again with start_index to read the next chunk.

Write the final answer in clear markdown. For every non-obvious factual claim, cite the source
inline as [short label](url) or add a "## Sources" section listing title + link for each page you used.
Do not invent URLs; only cite pages you actually fetched or that appeared in search tool results you then fetched.
"""


async def run_research(user_question: str) -> None:
    """
    Run a research agent using the MCP fetch server.
    """
    async with AsyncExitStack() as stack:
        fetch_server = await stack.enter_async_context(
            MCPServerStdio(params=fetch_params, client_session_timeout_seconds=SESSION_TIMEOUT)
        )
        agent = Agent(
            name="Agentic Researcher",
            instructions=research_instructions(),
            model=llm_model,
            mcp_servers=[fetch_server],
        )
        with trace("deep_link_research"):
            result = await Runner.run(agent, user_question, max_turns=30)
        display(Markdown(result.final_output))

In [ ]:
# You can replace the topic with any question you want to research
topic = (
    "What is the Model Context Protocol (MCP), and why do agents use it? "
    "Give a short overview with citations from fetched pages."
)

await run_research(topic)